<a href="https://www.kaggle.com/code/kirillgubkin/f1-pitstop-prediction?scriptVersionId=321776095" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv


In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/train.csv', index_col='id')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/test.csv', index_col='id')

In [3]:
train_df.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
id,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [4]:
test_df.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
id,,,,,,,,,,,,,,
439140,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
439141,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
439142,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
439143,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
439144,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0


In [5]:
print(train_df.shape)
print(test_df.shape)

(439140, 15)
(188165, 14)


In [6]:
train_df.isna().sum()

Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
dtype: int64

In [7]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 439140 entries, 0 to 439139
Data columns (total 15 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Driver                  439140 non-null  object 
 1   Compound                439140 non-null  object 
 2   Race                    439140 non-null  object 
 3   Year                    439140 non-null  int64  
 4   PitStop                 439140 non-null  int64  
 5   LapNumber               439140 non-null  int64  
 6   Stint                   439140 non-null  int64  
 7   TyreLife                439140 non-null  float64
 8   Position                439140 non-null  int64  
 9   LapTime (s)             439140 non-null  float64
 10  LapTime_Delta           439140 non-null  float64
 11  Cumulative_Degradation  439140 non-null  float64
 12  RaceProgress            439140 non-null  float64
 13  Position_Change         439140 non-null  float64
 14  PitNextLap              4

In [8]:
print(train_df['PitNextLap'].value_counts(normalize=True))

PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from category_encoders import TargetEncoder

train_df = train_df.sort_values(['Race', 'Driver', 'LapNumber'])
test_df = test_df.sort_values(['Race', 'Driver', 'LapNumber'])
features = ['Year', 'LapNumber', 'Stint', 'TyreLife', 'Position',
            'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
            'RaceProgress', 'Position_Change', 'Driver', 'Compound', 'Race']

params = {
    'random_state': 42,
    'class_weight': 'balanced',
    'n_estimators': 500,
    'learning_rate': 0.05,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'verbosity': -1
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(train_df, train_df['PitNextLap'])):
    print(f'Fold {fold+1}')
    
    train_fold = train_df.iloc[train_idx].copy()
    val_fold = train_df.iloc[val_idx].copy()
    
    te = TargetEncoder(cols=['Driver', 'Compound', 'Race'])
    train_fold[['Driver_te', 'Compound_te', 'Race_te']] = te.fit_transform(
        train_fold[['Driver', 'Compound', 'Race']], train_fold['PitNextLap']
    )
    val_fold[['Driver_te', 'Compound_te', 'Race_te']] = te.transform(
        val_fold[['Driver', 'Compound', 'Race']]
    )
    
    train_fold['TyreLife_lag1'] = train_fold.groupby(['Race', 'Driver'])['TyreLife'].shift(1)
    train_fold['TyreLife_lag2'] = train_fold.groupby(['Race', 'Driver'])['TyreLife'].shift(2)
    train_fold[['TyreLife_lag1', 'TyreLife_lag2']] = train_fold[['TyreLife_lag1', 'TyreLife_lag2']].fillna(0)
    
    val_fold['TyreLife_lag1'] = val_fold.groupby(['Race', 'Driver'])['TyreLife'].shift(1)
    val_fold['TyreLife_lag2'] = val_fold.groupby(['Race', 'Driver'])['TyreLife'].shift(2)
    val_fold[['TyreLife_lag1', 'TyreLife_lag2']] = val_fold[['TyreLife_lag1', 'TyreLife_lag2']].fillna(0)
    
    feature_cols = ['Year', 'LapNumber', 'Stint', 'TyreLife', 'TyreLife_lag1', 'TyreLife_lag2',
                    'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
                    'RaceProgress', 'Position_Change', 'Driver_te', 'Compound_te', 'Race_te']
    
    X_train_fold = train_fold[feature_cols]
    y_train_fold = train_fold['PitNextLap']
    X_val_fold = val_fold[feature_cols]
    y_val_fold = val_fold['PitNextLap']
    
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)], 
              eval_metric='auc', callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    
    preds_val = model.predict_proba(X_val_fold)[:, 1]
    auc = roc_auc_score(y_val_fold, preds_val)
    cv_scores.append(auc)
    print(f'Fold AUC: {auc:.4f}')

print(f'Mean CV AUC: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})')

Fold 1
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.94371	valid_0's binary_logloss: 0.310402
Fold AUC: 0.9437
Fold 2
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.944575	valid_0's binary_logloss: 0.307959
Fold AUC: 0.9446
Fold 3
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.945473	valid_0's binary_logloss: 0.307473
Fold AUC: 0.9455
Fold 4
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.943543	valid_0's binary_logloss: 0.305466
Fold AUC: 0.9435
Fold 5
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's auc: 0.943349	valid_0's binary_logloss: 0.310309
Fold AUC: 0.9433
Mean CV AUC

In [10]:
te_full = TargetEncoder(cols=['Driver', 'Compound', 'Race'])
train_df[['Driver_te', 'Compound_te', 'Race_te']] = te_full.fit_transform(
    train_df[['Driver', 'Compound', 'Race']], train_df['PitNextLap']
)

train_df['TyreLife_lag1'] = train_df.groupby(['Race', 'Driver'])['TyreLife'].shift(1)
train_df['TyreLife_lag2'] = train_df.groupby(['Race', 'Driver'])['TyreLife'].shift(2)
train_df[['TyreLife_lag1', 'TyreLife_lag2']] = train_df[['TyreLife_lag1', 'TyreLife_lag2']].fillna(0)

test_df = test_df.copy()

test_df[['Driver_te', 'Compound_te', 'Race_te']] = te_full.transform(
    test_df[['Driver', 'Compound', 'Race']]
)

test_df['TyreLife_lag1'] = test_df.groupby(['Race', 'Driver'])['TyreLife'].shift(1)
test_df['TyreLife_lag2'] = test_df.groupby(['Race', 'Driver'])['TyreLife'].shift(2)
test_df[['TyreLife_lag1', 'TyreLife_lag2']] = test_df[['TyreLife_lag1', 'TyreLife_lag2']].fillna(0)

feature_cols = ['Year', 'LapNumber', 'Stint', 'TyreLife', 'TyreLife_lag1', 'TyreLife_lag2',
                'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
                'RaceProgress', 'Position_Change', 'Driver_te', 'Compound_te', 'Race_te']

X_train_full = train_df[feature_cols]
y_train_full = train_df['PitNextLap']

final_model = lgb.LGBMClassifier(**params)  
final_model.fit(X_train_full, y_train_full)

X_test = test_df[feature_cols]
predictions = final_model.predict_proba(X_test)[:, 1]  


submission = pd.DataFrame({
    'id': test_df.index,
    'PitNextLap': predictions
})
submission.to_csv('submission.csv', index=False)


In [11]:
print(submission.head())

       id  PitNextLap
0  463410    0.272877
1  496081    0.062655
2  452309    0.464785
3  476804    0.068836
4  449143    0.451237
